In [ ]:
# # --- KONFIGURATION ---
# BASE_URL = 'https://api.openalex.org/authors'
# # Vi holder batchen på 20 for at undgå at URL'en bliver for lang pga. navnenes længde
# BATCH_SIZE = 20 
# all_author_data = []

# # Antag at 'keynote_speakers' er din liste med navne
# # keynote_speakers = ['Sarah Williams', 'John Doe', ...] 

# print(f"Starter bulk-søgning for {len(names)} navne...")

# # --- BULK LOOP ---
# for i in range(0, len(names), BATCH_SIZE):
#     batch_names = names[i : i + BATCH_SIZE]
    
#     # Vi samler navnene med | (OR)
#     # Vi bruger 'display_name' filteret i stedet for 'search' for at kunne lave bulk
#     name_filter = '|'.join(batch_names)
    
#     params = {
#         'filter': f'display_name:{name_filter}',
#         'select': 'id,display_name,works_count,summary_stats,last_known_institutions',
#         'per_page': 200, # Vi sætter den højt, da ét navn kan give flere resultater
#         'mailto': EMAIL
#     }
    
#     if API_KEY:
#         params['api_key'] = API_KEY

#     try:
#         response = requests.get(BASE_URL, params=params)
#         print(response)
        
#         if response.status_code == 200:
#             data = response.json()
#             results = data.get('results', [])
            
#             for author in results:
#                 # Samme logik til at udtrække data som før
#                 subset = {
#                     'id': author.get('id'),
#                     'display_name': author.get('display_name'),
#                     'works_count': author.get('works_count'),
#                     'h_index': author.get('summary_stats', {}).get('h_index'),
#                     'country_code': author.get('last_known_institutions', [{}])[0].get('country_code') if author.get('last_known_institutions') else None
#                 }
#                 all_author_data.append(subset)
            
#             print(f"Batch {i//BATCH_SIZE + 1} færdig. Fundet indtil videre: {len(all_author_data)}")
            
#         elif response.status_code == 429:
#             print("Rate limit ramt! Venter 1 sekund...")
#             time.sleep(1)
            
#     except Exception as e:
#         print(f"Fejl ved batch {i}: {e}")

# # --- SAMLING AF DATA ---
# df_final = pd.DataFrame(all_author_data)

# # Da ét navn (f.eks. "Sarah Williams") kan returnere 5 forskellige forfattere,
# # vil du måske kun beholde den med flest værker (mest sandsynlige match):
# if not df_final.empty:
#     df_final = df_final.sort_values('works_count', ascending=False).drop_duplicates('display_name')

# print(f"\nFærdig! Metadata hentet for {len(df_final)} unikke navne-matches.")

In [15]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
YOUR_KEY_2 = 'I7BU1yoA12r5RaYCvba8eC' # Ole Henriksen
YOUR_KEY = 'U2HIfMm7lzzOACIZH2GvPS' # Uffe
LINK = 'https://ic2s2-2025.org/program/'

#### Speaker scrape

In [16]:
######## TEMP ##########
## SPeaker scrape
r = requests.get(LINK)  
soup = BeautifulSoup(r.content)
# 1. Find the specific table containing the schedule
schedule_table = soup.find(id="summary")

keynote_speakers = []
# 2. Iterate through all stripped text strings within that table
# .stripped_strings is useful here because it handles the <br/> tags 
# that separate speakers in the same cell automatically.
if schedule_table:
    for text in schedule_table.stripped_strings:
        #print(text)
        if "Keynote:" in text:
            # 3. Clean the string to get just the name
            # Split by "Keynote:" and take the last part, then strip whitespace
            name = text.split("Keynote:")[-1].strip()
            keynote_speakers.append(name)

# Output the results
print("Found Keynote Speakers:")
for speaker in keynote_speakers:
    print(f"- {speaker}")

print(len(keynote_speakers))

Found Keynote Speakers:
- Dean Eckles
- Kathleen Carley
- Laura Nelson
- Duncan J. Watts
- Brandon Stewart
- Lisa P. Argyle
- Amir Goldberg
- Arnout van de Rijt
- Sarah Williams
9


#### df builder

In [17]:
# 1. Setup URL og Søgning

keynote_speakers
NAME = 'Sarah Williams'
BASE_URL = 'https://api.openalex.org/'
RESOURCE = 'authors'


# Vi kombinerer ikke navnet direkte i URL'en, men sender det som data
COMPLETE_URL = BASE_URL + RESOURCE


# 2. Definer parametre
# OpenAlex bruger 'search' parameteren til at lede i tekst
parameters = {
    'search': NAME,
    'api_key': YOUR_KEY}


seleceted_keys = ['id', 'display_name', 'works_count', 'summary_stats', 'works_api_url', 'last_known_institutions' ]

df = pd.DataFrame()
for speaker in keynote_speakers:
    parameters['search'] = speaker
    response = requests.get(COMPLETE_URL, params=parameters)
    dct = response.json()
    #print(dct)
    if dct['results']:
        results = dct['results'][0]
        subset = {}
        for key in seleceted_keys:
            if key == 'summary_stats':
                subset['h_index'] = results.get(key, {}).get('h_index')
            elif key == 'last_known_institutions':
                subset['country_code'] = results.get(key, [{}])[0].get('country_code') if results.get(key) else None
            else:
                subset[key] = results.get(key)
        df = pd.concat([df, pd.DataFrame([subset])], ignore_index=True)
df



{'meta': {'count': 17, 'db_response_time_ms': 25, 'page': 1, 'per_page': 25, 'groups_count': None, 'cost_usd': 0.001}, 'results': [{'id': 'https://openalex.org/A5080265907', 'orcid': 'https://orcid.org/0000-0001-8439-442X', 'display_name': 'Dean Eckles', 'display_name_alternatives': ['Dean Eckles', 'Dominick Shattuck', 'Eckles, Dean', 'Eckles, Dean (MIT); Kizilcec, René F. (Stanford University); Bakshy, Eytan (Facebook)', 'Karrer, Brian'], 'relevance_score': 12763.713, 'works_count': 121, 'cited_by_count': 5283, 'summary_stats': {'2yr_mean_citedness': 3.25, 'h_index': 34, 'i10_index': 59}, 'ids': {'openalex': 'https://openalex.org/A5080265907', 'orcid': 'https://orcid.org/0000-0001-8439-442X'}, 'affiliations': [{'institution': {'id': 'https://openalex.org/I102179633', 'ror': 'https://ror.org/02tvcev59', 'display_name': 'New School', 'country_code': 'US', 'type': 'education', 'lineage': ['https://openalex.org/I102179633']}, 'years': [2018]}, {'institution': {'id': 'https://openalex.org/

,id,display_name,works_count,h_index,works_api_url,country_code
0,https://openalex.org/A5080265907,Dean Eckles,121,34,https://api.openalex.org/works?filter=author.i...,US
1,https://openalex.org/A5085927300,Kathleen M. Carley,881,80,https://api.openalex.org/works?filter=author.i...,None
2,https://openalex.org/A5111667138,Laura E. Nelson,21,12,https://api.openalex.org/works?filter=author.i...,US
3,https://openalex.org/A5000679279,Duncan J. Watts,233,70,https://api.openalex.org/works?filter=author.i...,US
4,https://openalex.org/A5113226689,Brandon Stewart,305,31,https://api.openalex.org/works?filter=author.i...,US
5,https://openalex.org/A5070497645,Lisa P. Argyle,38,12,https://api.openalex.org/works?filter=author.i...,None
6,https://openalex.org/A5109219790,Amir Pnueli,367,88,https://api.openalex.org/works?filter=author.i...,IL
7,https://openalex.org/A5081982677,Arnout van de Rijt,106,23,https://api.openalex.org/works?filter=author.i...,IT
8,https://openalex.org/A5033458280,Sarah Williams,123,34,https://api.openalex.org/works?filter=author.i...,GB


## Snowballing

In [ ]:
# ... (Din forberedelse af df_filtered_authors og koncepter foroven er uændret) ...

# 1. RENS OG FORBERED ID-LISTE
# Sørg for at 'ids' kun indeholder ID'er (fx "A5080265907") og ikke hele URL'er
# --- KONFIGURATION ---
WORKING_URL = 'https://api.openalex.org/works'
EMAIL = "s245109@dtu.dk"  # Din email til Polite Pool
API_KEY = YOUR_KEY
# 2. DEFINER KONCEPTER (CSS FILTER)
social_ids = "C144024400|C15744967|C162324750|C17744445"
quant_ids = "C33923547|C121332964|C41008148"

# 3. OPSÆTNING AF API PARAMETRE
# Vi starter cursoren ved '*'
current_cursor = '*'
all_works = []
page_num = 1

# 1. FORBEREDELSE AF DATA
# Vi antager at 'df' findes. Hvis du tester uden 'df', så lav en dummy her:
# df = pd.DataFrame({'id': ['https://openalex.org/A5080265907'], 'works_count': [100]})

# Filtrer forfattere (5 til 5000 værker)
df_filtered_authors = df[(df['works_count'] >= 5) & (df['works_count'] <= 5000)].copy()
ids = df_filtered_authors['id'].astype(str).str.split('/').str[-1].tolist()


BATCH_SIZE = 25
all_works = [] # Denne skal ligge UDENFOR loops, så vi samler alt op

print(f"Starter indsamling for {len(ids)} forfattere i batches af {BATCH_SIZE}...")

# --- YDRE LOOP: BATCHING ---
# Vi bruger range() til at hoppe med 25 ad gangen (0, 25, 50...)
for i in range(0, len(ids), BATCH_SIZE):
    
    # Udvælg de næste 25 ID'er
    batch_ids = ids[i : i + BATCH_SIZE]
    batch_idstr = '|'.join(batch_ids)
    
    print(f"--- Behandler batch {i//BATCH_SIZE + 1} (Forfatter {i+1} til {min(i+BATCH_SIZE, len(ids))}) ---")

    # VIGTIGT: Cursor skal nulstilles for hver ny batch!
    current_cursor = '*'
    page_num = 1
    
    # Byg filter-strengen for DENNE batch
    filter_string = (
        f'author.id:{batch_idstr},'
        'authors_count:<10,'
        f'concepts.id:{social_ids},'
        f'concepts.id:{quant_ids},'
        'cited_by_count:>10'
    )

    # --- INDRE LOOP: API PAGINATION (Din eksisterende logik) ---
    while True:
        params = {
            'filter': filter_string,
            'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts',
            'per_page': 200,
            'cursor': current_cursor,
            'mailto': EMAIL 
        }
        
        if API_KEY:
            params['api_key'] = API_KEY

        try:
            response = requests.get(WORKING_URL, params=params)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                meta = data.get('meta', {})
                
                # Hvis listen er tom, er denne batch færdig -> Break indre loop
                if not results:
                    print(f"   Batch færdig. Ingen flere sider.")
                    break
                    
                # Gem resultater
                for work in results:
                    all_works.append({
                        'id': work.get('id'),
                        'publication_year': work.get('publication_year'),
                        'cited_by_count': work.get('cited_by_count'),
                        'title': work.get('title'),
                        'author_ids': [auth['author']['id'] for auth in work.get('authorships', [])],
                        'abstract_inverted_index': work.get('abstract_inverted_index')
                    })
                
                print(f"   Side {page_num} hentet ({len(results)} værker). Total indsamlet: {len(all_works)}")
                
                # Opdater cursor
                current_cursor = meta.get('next_cursor')
                page_num += 1
                
                if not current_cursor:
                    break
                    
            elif response.status_code == 429:
                print("   For mange forespørgsler (429). Venter 1 sekund...")
                time.sleep(1)
                continue 
                
            else:
                print(f"   Fejl opstod: {response.status_code} - {response.text}")
                break # Hopper ud af while-loopet (til næste batch)

        except Exception as e:
            print(f"   Kritisk fejl: {e}")
            break

# 5. RESULTAT BEHANDLING (Uændret)
print(f"\nHelt færdig! Hentet i alt {len(all_works)} værker.")
# ... Resten af din DataFrame kode ...

Starter indsamling for 9 forfattere i batches af 25...
--- Behandler batch 1 (Forfatter 1 til 9) ---
   Side 1 hentet (200 værker). Total indsamlet: 200
   Side 2 hentet (175 værker). Total indsamlet: 375
   Batch færdig. Ingen flere sider.

Helt færdig! Hentet i alt 375 værker.


In [ ]:
D1 = pd.DataFrame(all_works)
D2 = D1[['id', 'publication_year', 'cited_by_count', 'author_ids']]
D3 = D1[['id', 'title', 'abstract_inverted_index']]



In [ ]:
D1

,id,publication_year,cited_by_count,title,author_ids,abstract_inverted_index
0,https://openalex.org/W2095655043,2013,3006,Text as Data: The Promise and Pitfalls of Auto...,"[https://openalex.org/A5090665793, https://ope...","{'Politics': [0], 'and': [1, 9, 67, 96, 99, 10..."
1,https://openalex.org/W2147453867,2006,2126,Experimental Study of Inequality and Unpredict...,"[https://openalex.org/A5058815871, https://ope...","{'Hit': [0], 'songs,': [1], 'books,': [2], 'an..."
2,https://openalex.org/W2150375325,2007,1866,"Influentials, Networks, and Public Opinion For...","[https://openalex.org/A5000679279, https://ope...","{'A': [0], 'central': [1], 'idea': [2], 'in': ..."
3,https://openalex.org/W2096974619,2014,1830,Structural Topic Models for Open‐Ended Survey ...,"[https://openalex.org/A5082742221, https://ope...","{'Collection': [0], 'and': [1, 14, 37, 78, 97,..."
4,https://openalex.org/W2049607688,2006,1696,Empirical Analysis of an Evolving Social Network,"[https://openalex.org/A5066696247, https://ope...","{'Social': [0], 'networks': [1], 'evolve': [2]..."
...,...,...,...,...,...,...
370,https://openalex.org/W1602696812,2008,12,Change Detection in Social Networks,"[https://openalex.org/A5079997412, https://ope...","{'Abstract': [0], ':': [1], 'Social': [2], 'ne..."
371,https://openalex.org/W3037966274,2020,12,A Dataset of Fact-Checked Images Shared on Wha...,"[https://openalex.org/A5077142904, https://ope...","{'(Dataset': [0], 'paper).': [1], 'In': [2, 59..."
372,https://openalex.org/W4287773562,2020,12,Stance in Replies and Quotes (SRQ): A New Data...,"[https://openalex.org/A5050370761, https://ope...","{'Automated': [0], 'ways': [1], 'to': [2, 16, ..."
373,https://openalex.org/W431568915,2015,12,Dissecting the Spirit of Gezi: Influence vs. S...,"[https://openalex.org/A5086827245, https://ope...",None


### all unique authors

In [ ]:
org_id_set = set(ids)
D2['author_ids_set'] = D2['author_ids'].apply(set) # Laver set for at fjerne dupes
all_unique_ids = set().union(*D2['author_ids_set'])
print(len(all_unique_ids))
all_unique_ids = all_unique_ids - org_id_set

print(len(all_unique_ids))

lst_all_unique_ids = list(all_unique_ids)

466
466


### UPDATE INFO FOR D4

In [ ]:
import requests
import pandas as pd
import time

# --- 1. FORBEREDELSE AF ID-LISTE ---
# Vi tager din liste af unikke co-authors og sikrer, at de er i det rigtige format
# Vi fjerner URL-delen, så vi kun har selve ID'et (fx A50...)
clean_ids = [str(i).split('/')[-1] for i in lst_all_unique_ids]

BATCH_SIZE = 50  # Vi kan hente op til 50 forfattere ad gangen i ét filter-kald
all_author_metadata = []

print(f"Starter indsamling af metadata for {len(clean_ids)} co-authors...")

# --- 2. BATCH LOOP (Optimering) ---
for i in range(0, len(clean_ids), BATCH_SIZE):
    batch = clean_ids[i : i + BATCH_SIZE]
    batch_str = '|'.join(batch)
    
    # Vi bruger 'filter' i stedet for 'search' for at ramme de præcise ID'er
    params = {
        'filter': f'openalex_id:{batch_str}',
        'select': 'id,display_name,works_count,summary_stats,last_known_institutions',
        'per_page': BATCH_SIZE,
        'mailto': EMAIL
    }
    
    if API_KEY:
        params['api_key'] = API_KEY

    try:
        response = requests.get("https://api.openalex.org/authors", params=params)
        
        if response.status_code == 200:
            data = response.json()
            results = data.get('results', [])
            
            for author in results:
                # Udpakker data præcis som i din oprindelige kode
                subset = {
                    'id': author.get('id'),
                    'display_name': author.get('display_name'),
                    'works_count': author.get('works_count'),
                    'h_index': author.get('summary_stats', {}).get('h_index'),
                    'country_code': author.get('last_known_institutions', [{}])[0].get('country_code') if author.get('last_known_institutions') else None
                }
                all_author_metadata.append(subset)
            
            print(f"Batch {i//BATCH_SIZE + 1} færdig. Hentet: {len(all_author_metadata)}")
            
        elif response.status_code == 429:
            print("Rate limit ramt. Venter...")
            time.sleep(1)
            # Her burde man ideelt set gå et skridt tilbage i loopet, men for simpelhedens skyld:
            continue
            
    except Exception as e:
        print(f"Fejl ved batch {i}: {e}")

# --- 3. KONVERTER TIL DATAFRAME ---
D4 = pd.DataFrame(all_author_metadata)

print(f"\nFærdig! Metadata hentet for {len(D4)} forfattere.")
display(D4.head())

Starter indsamling af metadata for 458 co-authors...
Batch 1 færdig. Hentet: 50
Batch 2 færdig. Hentet: 100
Batch 3 færdig. Hentet: 150
Batch 4 færdig. Hentet: 200
Batch 5 færdig. Hentet: 250
Batch 6 færdig. Hentet: 300
Batch 7 færdig. Hentet: 350
Batch 8 færdig. Hentet: 400
Batch 9 færdig. Hentet: 450
Batch 10 færdig. Hentet: 458

Færdig! Metadata hentet for 458 forfattere.


,id,display_name,works_count,h_index,country_code
0,https://openalex.org/A5100338946,Huan Liu,898,102,CN
1,https://openalex.org/A5073544378,Luı́s A. Nunes Amaral,371,79,US
2,https://openalex.org/A5100323773,Wei Wei,364,24,CN
3,https://openalex.org/A5068923033,Nitin Agarwal,318,30,US
4,https://openalex.org/A5077429082,Orna Kupferman,314,48,IL


In [ ]:
# ... (Din forberedelse af df_filtered_authors og koncepter foroven er uændret) ...

# 1. RENS OG FORBERED ID-LISTE
# Sørg for at 'ids' kun indeholder ID'er (fx "A5080265907") og ikke hele URL'er
# --- KONFIGURATION ---
WORKING_URL = 'https://api.openalex.org/works'
EMAIL = "s245109@dtu.dk"  # Din email til Polite Pool
API_KEY = YOUR_KEY
# 2. DEFINER KONCEPTER (CSS FILTER)
social_ids = "C144024400|C15744967|C162324750|C17744445"
quant_ids = "C33923547|C121332964|C41008148"

# 3. OPSÆTNING AF API PARAMETRE
# Vi starter cursoren ved '*'
current_cursor = '*'
all_works = []
page_num = 1

# 1. FORBEREDELSE AF DATA
# Vi antager at 'df' findes. Hvis du tester uden 'df', så lav en dummy her:
# df = pd.DataFrame({'id': ['https://openalex.org/A5080265907'], 'works_count': [100]})

# Filtrer forfattere (5 til 5000 værker)
df_filtered_authors = D4[(D4['works_count'] >= 5) & (D4['works_count'] <= 5000)].copy()
ids = df_filtered_authors['id']#.astype(str).str.split('/').str[-1].tolist()


BATCH_SIZE = 25
all_works = [] # Denne skal ligge UDENFOR loops, så vi samler alt op

print(f"Starter indsamling for {len(ids)} forfattere i batches af {BATCH_SIZE}...")

# --- YDRE LOOP: BATCHING ---
# Vi bruger range() til at hoppe med 25 ad gangen (0, 25, 50...)
for i in range(0, len(ids), BATCH_SIZE):
    
    # Udvælg de næste 25 ID'er
    batch_ids = ids[i : i + BATCH_SIZE]
    batch_idstr = '|'.join(batch_ids)
    
    print(f"--- Behandler batch {i//BATCH_SIZE + 1} (Forfatter {i+1} til {min(i+BATCH_SIZE, len(ids))}) ---")

    # VIGTIGT: Cursor skal nulstilles for hver ny batch!
    current_cursor = '*'
    page_num = 1
    
    # Byg filter-strengen for DENNE batch
    filter_string = (
        f'author.id:{batch_idstr},'
        'authors_count:<10,'
        f'concepts.id:{social_ids},'
        f'concepts.id:{quant_ids},'
        'cited_by_count:>10'
    )

    # --- INDRE LOOP: API PAGINATION (Din eksisterende logik) ---
    while True:
        params = {
            'filter': filter_string,
            'select': 'id,publication_year,cited_by_count,authorships,title,abstract_inverted_index,concepts',
            'per_page': 200,
            'cursor': current_cursor,
            'mailto': EMAIL 
        }
        
        if API_KEY:
            params['api_key'] = API_KEY

        try:
            response = requests.get(WORKING_URL, params=params)
            
            if response.status_code == 200:
                data = response.json()
                results = data.get('results', [])
                meta = data.get('meta', {})
                
                # Hvis listen er tom, er denne batch færdig -> Break indre loop
                if not results:
                    print(f"   Batch færdig. Ingen flere sider.")
                    break
                    
                # Gem resultater
                for work in results:
                    all_works.append({
                        'id': work.get('id'),
                        'publication_year': work.get('publication_year'),
                        'cited_by_count': work.get('cited_by_count'),
                        'title': work.get('title'),
                        'author_ids': [auth['author']['id'] for auth in work.get('authorships', [])],
                        'abstract_inverted_index': work.get('abstract_inverted_index')
                    })
                
                print(f"   Side {page_num} hentet ({len(results)} værker). Total indsamlet: {len(all_works)}")
                
                # Opdater cursor
                current_cursor = meta.get('next_cursor')
                page_num += 1
                
                if not current_cursor:
                    break
                    
            elif response.status_code == 429:
                print("   For mange forespørgsler (429). Venter 1 sekund...")
                time.sleep(1)
                continue 
                
            else:
                print(f"   Fejl opstod: {response.status_code} - {response.text}")
                break # Hopper ud af while-loopet (til næste batch)

        except Exception as e:
            print(f"   Kritisk fejl: {e}")
            break

# 5. RESULTAT BEHANDLING (Uændret)
print(f"\nHelt færdig! Hentet i alt {len(all_works)} værker.")
# ... Resten af din DataFrame kode ...

Starter indsamling for 420 forfattere i batches af 25...
--- Behandler batch 1 (Forfatter 1 til 25) ---
   Side 1 hentet (200 værker). Total indsamlet: 200
   Side 2 hentet (200 værker). Total indsamlet: 400
   Side 3 hentet (200 værker). Total indsamlet: 600
   Side 4 hentet (52 værker). Total indsamlet: 652
   Batch færdig. Ingen flere sider.
--- Behandler batch 2 (Forfatter 26 til 50) ---
   Side 1 hentet (200 værker). Total indsamlet: 852
   Side 2 hentet (52 værker). Total indsamlet: 904
   Batch færdig. Ingen flere sider.
--- Behandler batch 3 (Forfatter 51 til 75) ---
   Side 1 hentet (200 værker). Total indsamlet: 1104
   Side 2 hentet (200 værker). Total indsamlet: 1304
   Side 3 hentet (200 værker). Total indsamlet: 1504
   Side 4 hentet (153 værker). Total indsamlet: 1657
   Batch færdig. Ingen flere sider.
--- Behandler batch 4 (Forfatter 76 til 100) ---
   Side 1 hentet (200 værker). Total indsamlet: 1857
   Side 2 hentet (160 værker). Total indsamlet: 2017
   Batch færdig

In [ ]:
D5 = pd.DataFrame(all_works)
print(D5.shape)
D5 = D5.drop_duplicates(subset= ['id'], keep= 'first')
print(D5.shape)
D6 = D5[['id', 'publication_year', 'cited_by_count', 'author_ids']]
D7 = D5[['id', 'title', 'abstract_inverted_index']]






(8200, 6)
(7443, 6)


In [ ]:
D5

NameError: name 'D5' is not defined